# Staged Evaluation Runner for PubMedQA (v0 smoke)

Loads PubMedQA cases, reveals `CONTEXTS` **one LABEL at a time**, calls Groq
`llama-3.1-8b-instant` at each stage with strict JSON output, dispatches mock
tools, and reports routing / revision / accuracy metrics. N=5 smoke run.

**Requires:** `GROQ_API_KEY` in a `.env` at the repo root (see `.env.example`).
Get a free key at <https://console.groq.com>.
**Data dir:** Assumed to be ../pubmedqa/


In [26]:
# One-time install (uncomment on first run).
# %pip install -q groq python-dotenv tenacity


In [1]:
import os, json, math, datetime, re
from dataclasses import dataclass, asdict, field
from typing import Optional

# If the notebook's cwd is staged_eval/, hop up to the repo root so
# relative paths like "pubmedqa/data/..." resolve consistently.
if os.path.basename(os.getcwd()) == "staged_eval":
    os.chdir("..")
print("cwd:", os.getcwd())

from dotenv import load_dotenv
from tenacity import retry, wait_exponential, stop_after_attempt
from groq import Groq

load_dotenv(override=True)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
GROQ_MODEL   = os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant")

if not GROQ_API_KEY:
    print("[WARN] GROQ_API_KEY not set. Cells that call the LLM (15, 17, 21) will fail. "
          "Add it to a `.env` at repo root before running the smoke.")
else:
    print(f"GROQ_API_KEY loaded (len={len(GROQ_API_KEY)}); model={GROQ_MODEL}")


cwd: /Users/davidman/Desktop/Research/Algoverse/MedRouteBench
GROQ_API_KEY loaded (len=56); model=llama-3.1-8b-instant


## Config


In [2]:
TEST_SET_PATH     = "../pubmedqa/data/test_set.json"          # relative to repo root
GROUND_TRUTH_PATH = "../pubmedqa/data/test_ground_truth.json"
RUNS_DIR          = "staged_eval/runs"
SMOKE_N           = 5
TEMPERATURE       = 0.0
MAX_TOKENS        = 512


## Action ontology + output schema

At every stage the LLM must return a single JSON object matching `AgentOutput`.
`validate(...)` never raises — on failure the runner attempts one repair prompt
before falling back to `ABSTAIN`.


In [3]:
ACTIONS = ["FOLLOW_UP", "REVISE_ANSWER", "ANSWER", "ABSTAIN"]
NONFINAL_ALLOWED = {"FOLLOW_UP", "REVISE_ANSWER", "ANSWER"}   # ABSTAIN forbidden non-final
FINAL_ALLOWED    = {"REVISE_ANSWER", "ANSWER", "ABSTAIN"}      # FOLLOW_UP forbidden final
VALID_ANSWERS    = {"yes", "no", "maybe", None}


@dataclass
class AgentOutput:
    action: str
    answer: Optional[str]
    confidence: float
    reason_for_action: str
    needed_information: Optional[str]


def safe_json_loads(s):
    try:
        return json.loads(s), None
    except Exception as e:
        return None, f"json_parse: {e}"


def validate(raw):
    """Return (AgentOutput, None) on success or (None, error_str) on failure. Never raises."""
    if not isinstance(raw, dict):
        return None, "root is not object"
    required = ["action", "answer", "confidence", "reason_for_action", "needed_information"]
    for k in required:
        if k not in raw:
            return None, f"missing key '{k}'"
    action = str(raw["action"]).strip().upper()
    if action not in ACTIONS:
        return None, f"invalid action '{action}'"
    ans = raw["answer"]
    if isinstance(ans, str):
        ans = ans.strip().lower() or None
    if ans not in VALID_ANSWERS:
        return None, f"invalid answer '{ans}'"
    try:
        conf = float(raw["confidence"])
    except Exception:
        return None, "confidence not a number"
    conf = max(0.0, min(1.0, conf))
    reason = raw["reason_for_action"]
    if not isinstance(reason, str) or not reason.strip():
        return None, "empty reason_for_action"
    need = raw["needed_information"]
    if need is not None and not isinstance(need, str):
        return None, "needed_information not str/null"
    return AgentOutput(action, ans, conf, reason.strip(), need), None


# quick self-check
_ok, _err = validate({
    "action": "ANSWER", "answer": "yes", "confidence": 0.8,
    "reason_for_action": "clear evidence", "needed_information": None,
})
assert _ok is not None and _err is None, (_ok, _err)
_bad, _err = validate({})
assert _bad is None and _err and "missing key" in _err, (_bad, _err)
print("validate() OK")


validate() OK


## Data loader + stage helpers

Each case gets `len(CONTEXTS) + 1` stages: stage 0 is question-only; stage `k`
adds the k-th `LABEL <NAME>: <context>` pair to the prompt.


In [4]:
def load_cases(path, limit=None):
    with open(path) as f:
        raw = json.load(f)
    items = [{"pmid": p, **case} for p, case in raw.items()]
    return items[:limit] if limit else items


def load_ground_truth(path):
    with open(path) as f:
        return json.load(f)


def _normalized_labels(case):
    """Length-aligned labels for CONTEXTS. Blank/duplicate labels get SECTION_<i> suffixes."""
    ctx = case["CONTEXTS"]
    lbls = list(case.get("LABELS") or [])
    out, seen = [], set()
    for i in range(len(ctx)):
        raw_lbl = lbls[i] if i < len(lbls) else ""
        lbl = (raw_lbl or "").strip().upper().replace(" ", "_") or f"SECTION_{i}"
        if lbl in seen:
            lbl = f"{lbl}_{i}"
        seen.add(lbl)
        out.append(lbl)
    return out


def n_stages(case):
    return len(case["CONTEXTS"]) + 1  # stage 0 = question only


def revealed_pairs(case, stage):
    """List of (label, context) revealed at this stage. stage 0 => []."""
    if stage <= 0:
        return []
    labels = _normalized_labels(case)
    ctx    = case["CONTEXTS"]
    k = min(stage, len(ctx))
    return list(zip(labels[:k], ctx[:k]))


def is_final_stage(case, stage):
    return stage == len(case["CONTEXTS"])


def warn_if_degenerate_labels(cases, gt):
    labels = {gt[c["pmid"]] for c in cases if c["pmid"] in gt}
    if len(labels) <= 1:
        print(f"[WARN] All {len(cases)} loaded cases share label(s) {labels}. "
              f"Metrics uninformative; sample is only for wiring.")


# quick self-check
_cases = load_cases(TEST_SET_PATH, limit=SMOKE_N)
_c0 = _cases[0]
assert n_stages(_c0) == len(_c0["CONTEXTS"]) + 1
_pairs = revealed_pairs(_c0, 1)
assert len(_pairs) == 1 and isinstance(_pairs[0][0], str) and isinstance(_pairs[0][1], str)
print(f"loaded {len(_cases)} cases; case0 pmid={_c0['pmid']} "
      f"n_contexts={len(_c0['CONTEXTS'])} labels={_normalized_labels(_c0)}")


loaded 5 cases; case0 pmid=12377809 n_contexts=3 labels=['AIMS', 'METHODS', 'RESULTS']


## Prompts (strict JSON contract)


In [31]:
SYSTEM_PROMPT = """You are a careful biomedical reasoning agent answering a PubMedQA question in stages. New CONTEXTS are revealed one LABEL at a time.

At every stage you MUST respond with a SINGLE JSON object and NOTHING else — no
prose, no markdown, no code fences.

Required keys:
  "action": one of ["FOLLOW_UP","REVISE_ANSWER","ANSWER","ABSTAIN"]
  "answer": "yes" | "no" | "maybe" | null
  "confidence": float in [0.0, 1.0]
  "reason_for_action": short string (<= 240 chars)
  "needed_information": short string or null

Action semantics:
- FOLLOW_UP: you need the next case LABEL revealed
- REVISE_ANSWER: change your prior answer based on new information
- ANSWER: give an answer
- ABSTAIN: refuse when evidence is genuinely insufficient

Stage-gated action rules (STRICT):
- On any NON-FINAL stage, FOLLOW_UP, REVISE_ANSWER, and ANSWER are allowed. If you choose FOLLOW_UP, you must leave "answer" as null. Use REVISE_ANSWER only when your prior stage's answer differs from your new answer; otherwise use ANSWER. ABSTAIN is FORBIDDEN until the final stage.
- On the FINAL stage (all case LABELs revealed) you MUST choose exactly one of
  {ANSWER, REVISE_ANSWER, ABSTAIN}. If you choose ANSWER or REVISE_ANSWER, the "answer" key MUST be filled with "yes", "no", or "maybe". If you choose ABSTAIN, you must leave the key as null. FOLLOW_UP is FORBIDDEN on the final stage.
"""


def build_user_prompt(case, stage, total_stages, prior_output):
    is_final = (stage == total_stages - 1)
    lines = [
        f"STAGE {stage} of {total_stages - 1} "
        f"(0 = question only, {total_stages - 1} = all LABELs revealed).",
        f"QUESTION:\n{case['QUESTION']}",
    ]
    pairs = revealed_pairs(case, stage)
    if pairs:
        lines.append("REVEALED SO FAR:")
        for lbl, ctx in pairs:
            lines.append(f"LABEL {lbl}: {ctx}")
    else:
        lines.append("REVEALED SO FAR: (none — question only)")
    if prior_output:
        lines.append(f"YOUR PRIOR OUTPUT: {json.dumps(prior_output)}")
    if is_final:
        lines.append(
            "THIS IS THE FINAL STAGE. You MUST choose action ∈ "
            "{ANSWER, REVISE_ANSWER, ABSTAIN}. If not ABSTAIN, `answer` MUST be "
            '"yes", "no", or "maybe" — never null. FOLLOW_UP is FORBIDDEN now.'
        )
    lines.append("Respond with the required JSON object only.")
    return "\n\n".join(lines)


REPAIR_TEMPLATE = (
    "Your previous response failed schema validation with error: {ERROR}. "
    "Return a corrected JSON object with the required keys. No prose.\n\n"
    "Original prompt:\n{ORIGINAL}"
)


# quick self-check
_p_mid = build_user_prompt(_cases[0], 1, n_stages(_cases[0]), None)
_p_end = build_user_prompt(_cases[0], n_stages(_cases[0]) - 1, n_stages(_cases[0]), None)
assert "LABEL " in _p_end and "FINAL STAGE" in _p_end
assert "FINAL STAGE" not in _p_mid
print("prompt final-stage gate: OK\n")
print(_p_end[-400:])


prompt final-stage gate: OK

ly 30% of controls. Both the changes in length and thickness of the m. puborectalis were significantly different (p<0.01, chi(2) test) in patients versus control subjects.

THIS IS THE FINAL STAGE. You MUST choose action ∈ {ANSWER, REVISE_ANSWER, ABSTAIN}. If not ABSTAIN, `answer` MUST be "yes", "no", or "maybe" — never null. FOLLOW_UP is FORBIDDEN now.

Respond with the required JSON object only.


## LLM client (Groq)

Uses `groq.Groq.chat.completions.create` with `response_format={"type":"json_object"}`
and exponential-backoff retry.


In [32]:
_groq_client = None


def groq_client():
    global _groq_client
    if _groq_client is None:
        if not GROQ_API_KEY:
            raise RuntimeError("GROQ_API_KEY not set. Add it to a `.env` at repo root.")
        _groq_client = Groq(api_key=GROQ_API_KEY)
    return _groq_client


@retry(wait=wait_exponential(multiplier=1, min=1, max=8), stop=stop_after_attempt(3))
def call_json(system, user):
    resp = groq_client().chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content


# smoke ping (only if key present)
if GROQ_API_KEY:
    _r = call_json("You return JSON.", 'Return the object {"ok": true} and nothing else.')
    print("call_json smoke:", _r[:200])
else:
    print("[skip] GROQ_API_KEY not set — skipping live smoke ping.")


call_json smoke: {
  "ok": true
}


## Per-case staged runner

For a case with N contexts, loops over `range(N + 1)`. On validation failure it
retries once with a repair prompt; a second failure records `ABSTAIN` with
`parse_error=True`. Tool observations are injected into the *next* stage.


In [33]:
def run_case(case, gt_label):
    total = n_stages(case)
    trace = {
        "pmid": case["pmid"],
        "gt": gt_label,
        "n_stages": total,
        "n_contexts": len(case["CONTEXTS"]),
        "stages": [],
    }
    prior_output = None

    for stage in range(total):
        user = build_user_prompt(case, stage, total, prior_output)
        raw = call_json(SYSTEM_PROMPT, user)
        parsed_raw, _ = safe_json_loads(raw)
        parsed, err = validate(parsed_raw)
        parse_error = False

        if err:  # one repair attempt
            raw2 = call_json(
                SYSTEM_PROMPT,
                REPAIR_TEMPLATE.format(ERROR=err, ORIGINAL=user),
            )
            parsed_raw2, _ = safe_json_loads(raw2)
            parsed, err2 = validate(parsed_raw2)
            raw = raw2
            if err2:
                parsed = AgentOutput(
                    "ABSTAIN", None, 0.0,
                    f"parse_error: {err2}", None,
                )
                parse_error = True

        trace["stages"].append({
            "stage": stage,
            "is_final": stage == total - 1,
            "revealed_labels": [lbl for lbl, _ in revealed_pairs(case, stage)],
            "raw": raw,
            "parsed": asdict(parsed),
            "parse_error": parse_error,
        })
        prior_output = asdict(parsed)

    return trace


## Metrics

Action oracle per stage:
- **nonfinal** stage → {FOLLOW_UP, REVISE_ANSWER, ANSWER} (ABSTAIN forbidden)
- **final** stage → {REVISE_ANSWER, ANSWER, ABSTAIN} (FOLLOW_UP forbidden)

Because stage counts vary per case, `action_accuracy_by_stage` reports both
`by_absolute_stage` (index → accuracy across cases that reached that index) and
`by_role` (nonfinal vs final).


In [34]:
def action_accuracy_by_stage(traces):
    per_abs = {}                                 # stage_idx -> [correct, total]
    per_role = {"nonfinal": [0, 0], "final": [0, 0]}
    for t in traces:
        for s in t["stages"]:
            role = "final" if s["is_final"] else "nonfinal"
            allowed = FINAL_ALLOWED if role == "final" else NONFINAL_ALLOWED
            per_abs.setdefault(s["stage"], [0, 0])
            per_abs[s["stage"]][1] += 1
            per_role[role][1]      += 1
            if s["parsed"]["action"] in allowed:
                per_abs[s["stage"]][0] += 1
                per_role[role][0]      += 1
    return {
        "by_absolute_stage": {k: (c / n if n else None) for k, (c, n) in sorted(per_abs.items())},
        "by_role":           {k: (c / n if n else None) for k, (c, n) in per_role.items()},
    }


def final_answer_accuracy(traces, gt):
    correct = total = 0
    for t in traces:
        ans = t["stages"][-1]["parsed"]["answer"]
        if ans in {"yes", "no", "maybe"}:
            total += 1
            if ans == gt.get(t["pmid"]):
                correct += 1
    return {"accuracy": (correct / total if total else 0.0),
            "committed": total, "n": len(traces)}


def revision_correctness(traces, gt):
    rev = corr = 0
    for t in traces:
        for i in range(1, len(t["stages"])):
            prev_a = t["stages"][i - 1]["parsed"]["answer"]
            new_a  = t["stages"][i]["parsed"]["answer"]
            if prev_a and new_a and prev_a != new_a:
                rev += 1
                if new_a == gt.get(t["pmid"]):
                    corr += 1
    return {"correctness": (corr / rev if rev else None), "n_revisions": rev}


def premature_answer_rate(traces, gt):
    """Fraction of cases where the agent committed a WRONG answer at a non-final stage."""
    prem = 0
    for t in traces:
        for s in t["stages"][:-1]:
            if s["parsed"]["action"] == "ANSWER" and s["parsed"]["answer"] != gt.get(t["pmid"]):
                prem += 1
                break
    return prem / len(traces) if traces else 0.0


def missed_revision_rate(traces, gt):
    missed = eligible = 0
    for t in traces:
        stages = t["stages"]
        if len(stages) < 2:
            continue
        prev = stages[-2]["parsed"]["answer"]
        final = stages[-1]["parsed"]
        if prev and prev != gt.get(t["pmid"]):
            eligible += 1
            if final["action"] != "REVISE_ANSWER" or final["answer"] == prev:
                missed += 1
    return {"rate": (missed / eligible if eligible else None), "n_eligible": eligible}


def abstention_rate(traces):
    if not traces:
        return 0.0
    return sum(1 for t in traces if t["stages"][-1]["parsed"]["action"] == "ABSTAIN") / len(traces)


def tool_metrics(traces):
    """No tools in this ontology — returns zeroed structure for backwards compat."""
    return {"call_rate": 0.0, "usefulness_proxy": None, "n_calls": 0}


# --- toy self-check with a hand-built trace ---
_toy = [{
    "pmid": "TEST", "gt": "yes", "n_stages": 3, "n_contexts": 2,
    "stages": [
        {"stage": 0, "is_final": False, "revealed_labels": [],
         "parsed": {"action": "FOLLOW_UP", "answer": None, "confidence": 0.3,
                    "reason_for_action": "need context", "needed_information": "results"},
         "parse_error": False, "raw": ""},
        {"stage": 1, "is_final": False, "revealed_labels": ["BACKGROUND"],
         "parsed": {"action": "ANSWER", "answer": "no", "confidence": 0.4,
                    "reason_for_action": "initial read", "needed_information": None},
         "parse_error": False, "raw": ""},
        {"stage": 2, "is_final": True, "revealed_labels": ["BACKGROUND", "RESULTS"],
         "parsed": {"action": "REVISE_ANSWER", "answer": "yes", "confidence": 0.85,
                    "reason_for_action": "results change conclusion", "needed_information": None},
         "parse_error": False, "raw": ""},
    ],
}]
_gt = {"TEST": "yes"}
print("action_accuracy_by_stage:", action_accuracy_by_stage(_toy))
print("final_answer_accuracy:   ", final_answer_accuracy(_toy, _gt))
print("revision_correctness:    ", revision_correctness(_toy, _gt))
print("premature_answer_rate:   ", premature_answer_rate(_toy, _gt))
print("missed_revision_rate:    ", missed_revision_rate(_toy, _gt))
print("abstention_rate:         ", abstention_rate(_toy))
print("tool_metrics:            ", tool_metrics(_toy))


action_accuracy_by_stage: {'by_absolute_stage': {0: 1.0, 1: 1.0, 2: 1.0}, 'by_role': {'nonfinal': 1.0, 'final': 1.0}}
final_answer_accuracy:    {'accuracy': 1.0, 'committed': 1, 'n': 1}
revision_correctness:     {'correctness': 1.0, 'n_revisions': 1}
premature_answer_rate:    1.0
missed_revision_rate:     {'rate': 0.0, 'n_eligible': 1}
abstention_rate:          0.0
tool_metrics:             {'call_rate': 0.0, 'usefulness_proxy': None, 'n_calls': 0}


## Smoke run (N=5)

Writes `staged_eval/runs/<UTC-ts>/trace_<pmid>.json` per case plus `report.json`.
Requires `GROQ_API_KEY`.


In [35]:
# --- Smoke config (override SMOKE_N inline; set to None to fall back to config) ---
N_RUN     = 50           # target sample size
STRATIFY  = True         # proportional yes/no/maybe from test_ground_truth.json
# ------------------------------------------------------------------------------------

all_cases = load_cases(TEST_SET_PATH)
gt        = load_ground_truth(GROUND_TRUTH_PATH)

if STRATIFY:
    # Proportional stratified sample by GT label, deterministic (file-order per bucket).
    from collections import Counter
    counts = Counter(gt.values())
    total_pop = sum(counts.values())
    # per-label quota, floor + largest-remainder to hit exactly N_RUN
    raw = {lbl: N_RUN * counts[lbl] / total_pop for lbl in counts}
    quotas = {lbl: int(v) for lbl, v in raw.items()}
    rem = N_RUN - sum(quotas.values())
    for lbl, _ in sorted(raw.items(), key=lambda kv: -(kv[1] - int(kv[1])))[:rem]:
        quotas[lbl] += 1
    picked, taken = [], {lbl: 0 for lbl in quotas}
    for c in all_cases:
        lbl = gt.get(c["pmid"])
        if lbl in quotas and taken[lbl] < quotas[lbl]:
            picked.append(c); taken[lbl] += 1
        if len(picked) == N_RUN:
            break
    cases = picked
    print(f"stratified sample: {taken} (target quotas = {quotas})")
else:
    cases = all_cases[:N_RUN]

warn_if_degenerate_labels(cases, gt)

ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = os.path.join(RUNS_DIR, ts)
os.makedirs(run_dir, exist_ok=True)
print("run_dir:", run_dir, "| n_cases:", len(cases))

traces = []
for i, case in enumerate(cases, 1):
    print(f"[{i}/{len(cases)}] pmid={case['pmid']} gt={gt.get(case['pmid'])} n_stages={n_stages(case)}")
    trace = run_case(case, gt.get(case["pmid"]))
    traces.append(trace)
    with open(os.path.join(run_dir, f"trace_{case['pmid']}.json"), "w") as f:
        json.dump(trace, f, indent=2, ensure_ascii=False)

report = {
    "model": GROQ_MODEL,
    "n_cases": len(traces),
    "sample_label_counts": dict(Counter(t["gt"] for t in traces)) if STRATIFY else None,
    "action_accuracy_by_stage": action_accuracy_by_stage(traces),
    "final_answer_accuracy":    final_answer_accuracy(traces, gt),
    "revision_correctness":     revision_correctness(traces, gt),
    "premature_answer_rate":    premature_answer_rate(traces, gt),
    "missed_revision_rate":     missed_revision_rate(traces, gt),
    "abstention_rate":          abstention_rate(traces),
    "tool_metrics":             tool_metrics(traces),
}
with open(os.path.join(run_dir, "report.json"), "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))


stratified sample: {'yes': 28, 'no': 17, 'maybe': 5} (target quotas = {'yes': 28, 'no': 17, 'maybe': 5})
run_dir: staged_eval/runs/20260705T022413Z | n_cases: 50
[1/50] pmid=12377809 gt=yes n_stages=4
[2/50] pmid=26163474 gt=yes n_stages=4
[3/50] pmid=19100463 gt=yes n_stages=4
[4/50] pmid=18537964 gt=yes n_stages=4
[5/50] pmid=12913878 gt=yes n_stages=4
[6/50] pmid=12765819 gt=yes n_stages=3
[7/50] pmid=25475395 gt=yes n_stages=5
[8/50] pmid=19130332 gt=yes n_stages=4
[9/50] pmid=9427037 gt=yes n_stages=4
[10/50] pmid=24481006 gt=yes n_stages=3
[11/50] pmid=8165771 gt=yes n_stages=4
[12/50] pmid=22680064 gt=yes n_stages=4
[13/50] pmid=22540518 gt=yes n_stages=4
[14/50] pmid=20629769 gt=yes n_stages=7
[15/50] pmid=21726930 gt=yes n_stages=4
[16/50] pmid=21481154 gt=yes n_stages=7
[17/50] pmid=22902073 gt=yes n_stages=4
[18/50] pmid=26370095 gt=yes n_stages=7
[19/50] pmid=18041059 gt=yes n_stages=4
[20/50] pmid=15041506 gt=yes n_stages=4
[21/50] pmid=11146778 gt=yes n_stages=7
[22/50] p

## Inspect a trace (optional)


In [36]:
# Peek at the first trace's per-stage actions/answers.
for s in traces[0]["stages"]:
    p = s["parsed"]
    print(f'stage={s["stage"]:>1} action={p["action"]:<18} answer={str(p["answer"]):<6} '
          f'conf={round(p["confidence"] or 0.0, 2):<4} tool={"Y" if s["tool_call"] else "-"}')


KeyError: 'tool_call'